In [1]:
import os
import random
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.eval.losses import QLIKELoss
from src.eval.scorer import VolatilityScorer
from src.models.causal_transformer_wrapper import CausalTransformerModel
from src.trainer.walk_forward import WalkForwardTrainer

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

OUT_PATH = PROJECT_ROOT / "data" / "processed"
RESULTS_PATH = PROJECT_ROOT / "results" / "Causal_Transformer"
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

df_train = pd.read_parquet(OUT_PATH / "brent_train.parquet")
y_train = df_train["log_returns"].values.astype("float64")
rs_train = df_train["rs_daily"].values.astype("float64")

df_test = pd.read_parquet(OUT_PATH / "brent_test.parquet")
y_test = df_test["log_returns"].values.astype("float64")
rs_test = df_test["rs_daily"].values.astype("float64")

assert not np.isnan(y_train).any() and not np.isnan(rs_train).any()
assert not np.isnan(y_test).any() and not np.isnan(rs_test).any()

print("=" * 70)
print(f"Train: {len(y_train)} obs  ({df_train.index[0].date()} -> {df_train.index[-1].date()})")
print(f"Test:  {len(y_test)} obs  ({df_test.index[0].date()} -> {df_test.index[-1].date()})")
print("=" * 70)

Train: 2171 obs  (2014-01-03 -> 2022-11-03)
Test:  953 obs  (2022-11-07 -> 2026-08-21)


In [2]:
import time

scorer = VolatilityScorer()

best_model = CausalTransformerModel(
    look_back=21,
    d_model=32,
    nhead=4,
    num_layers=2,
    dim_feedforward=64,
    dropout=0.1,
    activation="gelu",
    learning_rate=1e-3,
    weight_decay=1e-4,
    max_epochs=200,
    patience=20,
    batch_size=128,
    retrain_freq=1,
    seed=42,
)

start_time = time.time()
best_model.fit(y_train, rs_train)
end_time = time.time()

elapsed_time = end_time - start_time
print(f">> fit OK en {elapsed_time:.2f} s")
print("   diagnostics:", best_model.get_diagnostics())

n_oos = min(2, len(y_test))
y_test_20 = y_test[-n_oos:]
rs_test_20 = rs_test[-n_oos:]

t0_oos = time.time()
preds_test_20 = best_model.forecast(y_test_20, rs_seq=rs_test_20, debug=False)
oos_time = time.time() - t0_oos

print(
    f">> forecast (últimos {n_oos} días) OK en {oos_time:.2f} s "
    f"| shape={preds_test_20.shape}"
)

>> fit OK en 52.94 s
   diagnostics: {'converged': True, 'epochs_run': 27}
>> forecast (últimos 2 días) OK en 295.66 s | shape=(2,)


In [3]:
import numpy as np
from src.utils import trainer_toolkit as tt

history = best_model.get_history()
diagnostics = best_model.get_diagnostics()

train_loss = np.asarray(history.get("train_loss", []), dtype=float)
val_loss = np.asarray(history.get("val_loss", []), dtype=float)

assert len(train_loss) > 0 and len(val_loss) > 0, "El historial de entrenamiento está vacío."
assert np.isfinite(train_loss).all(), "train_loss contiene NaN o Inf."
assert np.isfinite(val_loss).all(), "val_loss contiene NaN o Inf."

print(
    f"Épocas ejecutadas: {diagnostics.get('epochs_run', len(train_loss))} "
    f"| Convergió: {diagnostics.get('converged')}"
)
print(
    f"Train QLIKE inicial -> final/mínimo: {train_loss[0]:.6f} -> {train_loss[-1]:.6f} "
    f"(min={train_loss.min():.6f})"
)
print(
    f"Val   QLIKE inicial -> final/mínimo: {val_loss[0]:.6f} -> {val_loss[-1]:.6f} "
    f"(min={val_loss.min():.6f})"
)

max_epochs_plotted = len(train_loss)

tt.plot_learning_curve(
    history=history,
    title="Curva de Aprendizaje (Causal Transformer)",
    ylab="Quasi-Likelihood (QLIKE)",
    xlab_pad=14,
    ylab_pad=14,
    line_width=2.0,
    title_size=14,
    label_size=14,
    tick_size=12,
    legend_size=14,
    ylim=(0, 5),
    xlim=(0, max(10, max_epochs_plotted)),
    y_step=0.5,
    x_step=5,
    legend_bbox=(0.95, 0.95),
)

Épocas ejecutadas: 27 | Convergió: True
Train QLIKE inicial -> final/mínimo: 5.394368 -> 0.369361 (min=0.369361)
Val   QLIKE inicial -> final/mínimo: 3.758891 -> 0.563701 (min=0.465299)


In [6]:
import numpy as np
import torch
from torch.utils.data import DataLoader

from src.data.dataset import TimeSeriesDataset
from src.utils import test_toolkit as ts

best_model.net.eval()
look_back = best_model.context_length

y_train_scaled = best_model.scaler.transform(y_train.reshape(-1, 1)).flatten()
rs_train_scaled = rs_train * best_model.target_scale

train_eval_dataset = TimeSeriesDataset(
    y_train_scaled,
    rs_train_scaled,
    look_back,
)
train_eval_loader = DataLoader(
    train_eval_dataset,
    batch_size=best_model.batch_size,
    shuffle=False,
)

preds_train_list = []
with torch.no_grad():
    for x_batch, _ in train_eval_loader:
        x_batch = x_batch.to(best_model.device)
        preds_batch = best_model.net(x_batch).squeeze(-1).detach().cpu().numpy()
        preds_train_list.append(preds_batch)

preds_train_var = np.concatenate(preds_train_list) / best_model.target_scale
n_preds = len(preds_train_var)

# Alinear índices según cuántos objetivos construya TimeSeriesDataset
start_idx = len(y_train) - n_preds
dates_aligned = df_train.index[start_idx : start_idx + n_preds]
rs_volatilidad_aligned = np.sqrt(np.maximum(rs_train[start_idx : start_idx + n_preds], 0.0))
pred_volatilidad = np.sqrt(np.maximum(preds_train_var, 0.0))

mask_crisis = (dates_aligned >= "2020-01-01") & (dates_aligned <= "2020-06-30")

ts.plot_volatility_fit(
    dates=dates_aligned,
    rs_variance=rs_volatilidad_aligned,
    pred_variance=pred_volatilidad,
    title="Ajuste dentro de la Muestra de Entrenamiento (Causal Transformer)",
    ylab="Volatilidad",
    label_realized=r"Volatilidad Realizada $\left(\sigma_R\right)$",
    label_pred=r"Volatilidad Causal Transformer $\left(\hat{\sigma}_t\right)$",
    ylim=(-0.01, 0.7),
    legend_bbox=(0.01, 0.98),
    zoom_slice=mask_crisis,
    zoom_pad_frac=0.04,
    zoom_title="Zoom: Crisis COVID-19",
    zoom_bbox=(0.18, 0.35, 0.40, 0.40),
    title_size=14,
    label_size=14,
    tick_size=12,
    legend_size=14,
)